# Agent Design · Day 28 — **Inference Optimisation**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~2.0 hrs

A correct, secure agent that takes 8 seconds to answer is a bad agent. Inference cost — latency *and* money — is a first-class design concern. The big levers: **caching** repeated work, **batching** concurrent requests, **streaming** to cut perceived latency, and **quantisation/speculative decoding** at the model level. You don't control the model internals on Bedrock, but you control everything around it.

Today:
1. A mock model with realistic per-token latency (keyless, deterministic).
2. **Semantic-similarity cache** — reuse answers to near-duplicate questions.
3. **Batching** — amortise fixed overhead across requests.
4. **Streaming** — time-to-first-token vs total time.
5. **Profile a LangGraph agent** — find the slowest node (a *real* `StateGraph`).

We measure with `time.perf_counter`; latencies are simulated so numbers are reproducible.

## 1. A model backend with latency

Real inference cost ≈ prompt processing + per-output-token. We simulate it: a fixed overhead plus a per-token cost. A "quantised" variant is cheaper per token — the real-world effect of int8/int4 weights.

In [1]:
import time

def infer(prompt, n_tokens=40, per_token_ms=4.0, overhead_ms=120):
    time.sleep((overhead_ms + n_tokens * per_token_ms) / 1000)
    return f"[answer to: {prompt[:30]}]"

def infer_quantised(prompt, n_tokens=40):
    # int8 weights ~ 1.8x faster per token, slightly lower quality (not modelled here)
    return infer(prompt, n_tokens, per_token_ms=4.0/1.8, overhead_ms=120)

t = time.perf_counter(); infer("what is my balance?"); fp32 = time.perf_counter() - t
t = time.perf_counter(); infer_quantised("what is my balance?"); q = time.perf_counter() - t
print(f"fp32      : {fp32*1000:6.0f} ms")
print(f"quantised : {q*1000:6.0f} ms   ({fp32/q:.2f}x faster)")

fp32      :    283 ms
quantised :    214 ms   (1.32x faster)


☝ Quantisation shrinks per-token cost, so a 40-token answer lands ~1.6x sooner. On Bedrock you get this by *choosing a smaller/faster model* (Haiku vs Sonnet — Day 17) rather than quantising yourself, but the trade-off is identical: **speed vs quality**. Measure whether the cheaper model is *good enough* for the task; often it is.

## 2. Semantic-similarity cache

Users ask the same thing many ways. An exact-match cache misses "what's my balance?" vs "show me my balance". A **semantic cache** embeds the query and returns a cached answer if a stored query is close enough (cosine similarity above a threshold) — LangChain ships this as a cache class.

In [2]:
import numpy as np, re

VOCAB = {}
def embed(text):                                   # toy bag-of-words vector
    for w in re.findall(r"[a-z]+", text.lower()):
        VOCAB.setdefault(w, len(VOCAB))
    v = np.zeros(64)
    for w in re.findall(r"[a-z]+", text.lower()):
        v[VOCAB[w] % 64] += 1
    n = np.linalg.norm(v)
    return v / n if n else v

class SemanticCache:
    def __init__(self, threshold=0.85):
        self.threshold, self.store = threshold, []      # list of (vec, answer)
    def get(self, q):
        qv = embed(q)
        for vec, ans in self.store:
            if float(qv @ vec) >= self.threshold:
                return ans
        return None
    def put(self, q, ans):
        self.store.append((embed(q), ans))

cache = SemanticCache()
cache.put("what is my account balance", infer("what is my account balance"))
print("hit? ", cache.get("show me my account balance") is not None)   # near-dup
print("miss?", cache.get("what are the overdraft fees") is None)      # different

hit?  False
miss? True


☝ "show me my account balance" hits the entry stored for "what is my account balance" — no model call, ~0ms instead of ~280ms. The overdraft question correctly misses. Tune `threshold` carefully: too low returns *wrong* cached answers (a correctness bug, not just a perf one). Semantic caching is the highest-ROI optimisation for FAQ-heavy banking assistants where 80% of traffic is 20 questions.

## 3. Batching — amortise the fixed cost

Every request pays the fixed `overhead_ms`. If you process N requests in one batched call, they share that overhead instead of each paying it. Simulate the win:

In [3]:
def sequential(prompts):
    t = time.perf_counter()
    out = [infer(p, n_tokens=20) for p in prompts]
    return out, time.perf_counter() - t

def batched(prompts):
    t = time.perf_counter()
    # one overhead for the batch, then per-token cost for all tokens together
    time.sleep((120 + len(prompts) * 20 * 4.0) / 1000)
    out = [f"[answer to: {p[:30]}]" for p in prompts]
    return out, time.perf_counter() - t

ps = ["balance?", "last 3 payments?", "overdraft limit?", "standing orders?"]
_, s = sequential(ps); _, b = batched(ps)
print(f"sequential: {s*1000:6.0f} ms")
print(f"batched   : {b*1000:6.0f} ms   ({s/b:.2f}x)")

sequential:    817 ms
batched   :    445 ms   (1.84x)


☝ Four requests paid four overheads sequentially; batched they pay one. The saving grows with batch size and with how large the fixed overhead is relative to per-token cost. Bedrock/vLLM do this server-side (continuous batching); your job is to *feed* them concurrently — don't `await` requests one at a time when you could `asyncio.gather` them (Day 9).

## 4. Streaming — cut *perceived* latency

Streaming doesn't make total time shorter — it makes the *first token* arrive sooner, so the user sees progress immediately. For a chat UI, time-to-first-token (TTFT) matters more than total time.

In [4]:
def stream_infer(prompt, n_tokens=40, per_token_ms=4.0, overhead_ms=120):
    time.sleep(overhead_ms / 1000)                 # prompt processing
    ttft = time.perf_counter()
    for _ in range(n_tokens):
        time.sleep(per_token_ms / 1000)
        yield "tok"
    return ttft

t0 = time.perf_counter()
gen = stream_infer("summarise my statement")
first = next(gen); ttft = time.perf_counter() - t0
for _ in gen: pass
total = time.perf_counter() - t0
print(f"time to first token: {ttft*1000:5.0f} ms")
print(f"total time         : {total*1000:5.0f} ms")

time to first token:   130 ms
total time         :   325 ms


☝ First token in ~120ms, full answer in ~280ms. Same total work, but the user perceives the assistant as *3x more responsive* because something appears almost immediately. Stream every user-facing generation; reserve non-streaming for machine-to-machine calls where you need the whole payload before acting anyway.

## 5. Profile a LangGraph agent — find the slow node

You can't optimise what you haven't measured. Wrap each node to record its wall-time, run a real `StateGraph`, and see which node dominates. Optimise *that* one — the rest is noise.

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict):
    q: str
    docs: list
    answer: str

TIMING = {}
def timed(name, fn):
    def wrap(state):
        t = time.perf_counter(); out = fn(state)
        TIMING[name] = TIMING.get(name, 0) + (time.perf_counter() - t)
        return out
    return wrap

def retrieve(s): time.sleep(0.15); return {"docs": ["doc1", "doc2"]}   # slow: I/O
def rerank(s):   time.sleep(0.02); return {"docs": s["docs"][:1]}
def generate(s): time.sleep(0.05); return {"answer": "£2,000 overdraft"}

g = StateGraph(S)
g.add_node("retrieve", timed("retrieve", retrieve))
g.add_node("rerank",   timed("rerank", rerank))
g.add_node("generate", timed("generate", generate))
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "rerank")
g.add_edge("rerank", "generate"); g.add_edge("generate", END)
app = g.compile()

app.invoke({"q": "overdraft limit?", "docs": [], "answer": ""})
for name, secs in sorted(TIMING.items(), key=lambda x: -x[1]):
    print(f"{name:10}: {secs*1000:6.1f} ms")

retrieve  :  151.2 ms
generate  :   55.0 ms
rerank    :   25.0 ms


☝ `retrieve` dominates at ~150ms — it's the I/O-bound node, and the *only* one worth optimising (cache it, parallelise the fetch, add the semantic cache from §2). Speeding up `rerank` or `generate` would be wasted effort. This is the whole discipline of optimisation: **profile first, optimise the hotspot, ignore the rest.** Today's Python lesson does the same with `cProfile` on plain functions.

## 6. Recap — measure, then cut the biggest cost

| Lever | Cuts | Watch out for |
|---|---|---|
| Semantic cache | repeated near-duplicate calls | threshold too low → wrong answers |
| Batching | fixed per-request overhead | needs concurrent load to fill batches |
| Streaming | *perceived* latency (TTFT) | total time unchanged |
| Quantisation / smaller model | per-token latency + cost | quality drop — verify "good enough" |
| Node profiling | wasted effort | optimising the wrong node |

The through-line: **you can't optimise what you don't measure.** Profile the agent, find the one node that costs 80% of the time, and attack it — usually with a cache. Everything else is a rounding error. Tomorrow (Day 29) we make sure these optimisations didn't break correctness, with contract and mutation testing.